In [1]:
from pathlib import Path
from urllib.request import urlopen
from datetime import datetime, timezone

BAS = "https://opendata-download-metobs.smhi.se/api/version/1.0"
STATION = 71420  # Göteborg A – bara för att se formatet
url = f"{BAS}/parameter/21/station/{STATION}/period/corrected-archive/data.csv"

# Spara i data/raw (ligger utanför git enligt fas 0)
ut = Path("data/raw") / f"smhi_p21_{STATION}_corrected-archive.csv"
ut.parent.mkdir(parents=True, exist_ok=True)

# Hämta bara om filen inte redan finns (SMHI vill inte ha onödiga anrop)
if ut.exists():
    print("Finns redan lokalt, hämtar inte om:", ut)
else:
    with urlopen(url) as r:
        ut.write_bytes(r.read())
    print("Hämtad", datetime.now(timezone.utc).isoformat(), "->", ut)

print(f"{ut.stat().st_size / 1e6:.1f} MB")

Finns redan lokalt, hämtar inte om: data\raw\smhi_p21_71420_corrected-archive.csv
6.6 MB


Skriver ut de första 20 raderna precis som de ser ut, utan att tolka något

In [2]:

with open(ut, encoding="utf-8", errors="replace") as f:
    for i, rad in enumerate(f):
        print(i, rad.rstrip())
        if i >= 19:
            break

0 ﻿Stationsnamn;Stationsnummer;Stationsnät;Mäthöjd (meter över marken)
1 Göteborg A;71420;SMHIs stationsnät;10.0
2 
3 Parameternamn;Beskrivning;Enhet
4 Byvind;max, 1 gång/tim;meter per sekund
5 
6 Tidsperiod (fr.o.m);Tidsperiod (t.o.m);Höjd (meter över havet);Latitud (decimalgrader);Longitud (decimalgrader)
7 1995-08-01 00:00:00;1995-09-30 23:59:59;23.0;57.7084;11.9939
8 1995-10-01 00:00:00;1998-05-31 23:59:59;5.0;57.7080;11.9938
9 1999-03-01 00:00:00;2026-11-04 23:21:21;3.038;57.7156;11.9924
10 
11 Datum;Tid (UTC);Byvind;Kvalitet;;Tidsutsnitt:
12 1995-08-04;11:00:00;0.1;Y;;Kvalitetskontrollerade historiska data (utom de senaste 3 mån)
13 1995-08-04;15:00:00;0.1;Y;;Tidsperiod (fr.o.m.) = 1995-08-01 00:00:00 (UTC)
14 1995-08-04;16:00:00;0.1;Y;;Tidsperiod (t.o.m.) = 2026-08-01 06:00:00 (UTC)
15 1995-08-04;21:00:00;2.5;Y;;Samplingstid = 1 timme
16 1995-08-04;22:00:00;2.2;Y;;
17 1995-08-04;23:00:00;2.5;Y;;Kvalitetskoderna:
18 1995-08-05;00:00:00;2.0;Y;;Grön (G) = Kontrollerade och godkända

In [3]:
ut = Path("../data/raw") / f"smhi_p21_{STATION}_corrected-archive.csv"

In [7]:
import pandas as pd

# Hitta rubrikraden. Antalet metadatarader skiljer sig mellan stationer,
# så vi letar i stället för att hårdkoda radnumret.
with open(ut, encoding="utf-8-sig") as f:
    for nr, rad in enumerate(f):
        if rad.startswith("Datum;"):
            break
print("Rubrikraden är rad", nr)

# Läs bara de fyra datakolumnerna; utf-8-sig tar bort BOM-tecknet
df = pd.read_csv(ut, sep=";", skiprows=nr, usecols=[0, 1, 2, 3],
                 encoding="utf-8-sig")
df.columns = ["datum", "tid", "byvind", "kvalitet"]

# Slå ihop datum och tid till en tidsstämpel i UTC och använd den som index
df["tidpunkt"] = pd.to_datetime(df["datum"] + " " + df["tid"], utc=True)
df = df.set_index("tidpunkt")[["byvind", "kvalitet"]]

print(len(df), "rader")
print(df.index.min(), "->", df.index.max())
print(df["kvalitet"].value_counts(normalize=True).round(3))
df.head()

Rubrikraden är rad 11
254288 rader
1995-08-04 11:00:00+00:00 -> 2026-08-01 06:00:00+00:00
kvalitet
Y    0.559
G    0.441
Name: proportion, dtype: float64


,byvind,kvalitet
tidpunkt,,
1995-08-04 11:00:00+00:00,0.1,Y
1995-08-04 15:00:00+00:00,0.1,Y
1995-08-04 16:00:00+00:00,0.1,Y
1995-08-04 21:00:00+00:00,2.5,Y
1995-08-04 22:00:00+00:00,2.2,Y


In [8]:
nominella_timmar = (df.index.max() - df.index.min()) / pd.Timedelta(hours=1) + 1
print(nominella_timmar, len(df) / nominella_timmar)

271676.0 0.9359972908906197


In [11]:
# 1. Täckning: andelen timmar som faktiskt har ett värde
nominella = (df.index.max() - df.index.min()) / pd.Timedelta(hours=1) + 1
print("Täckning:", round(len(df) / nominella, 3))

# 2a. Andel Y per år: är flaggan samlad i tiden?
y = df["kvalitet"] == "Y"
print(y.groupby(df.index.year).mean().round(2).to_string())

# 2b. Andel Y bland de högsta 0,1 % jämfört med alla värden
topp = df["byvind"] >= df["byvind"].quantile(0.999)
print("Y bland alla:", round(y.mean(), 3),
      "| Y bland topp 0,1 %:", round(y[topp].mean(), 3),
      "| antal toppvärden:", topp.sum())

Täckning: 0.936
tidpunkt
1995    1.0
1996    1.0
1997    1.0
1998    1.0
1999    1.0
2000    1.0
2001    1.0
2002    1.0
2003    1.0
2004    1.0
2005    1.0
2006    1.0
2007    1.0
2008    1.0
2009    1.0
2010    1.0
2011    1.0
2012    1.0
2013    0.0
2014    0.0
2015    0.0
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0
2025    0.0
2026    0.0
Y bland alla: 0.559 | Y bland topp 0,1 %: 0.783 | antal toppvärden: 267


Skiljer sig andel Y markerade datapunkter i 0,1% kvantilen för och efter 2013?

In [12]:
pre2013 = df.index.year < 2013
andel_tid_pre2013 = pre2013.mean()
andel_topp_pre2013 = topp[pre2013].sum() / topp.sum()
print("Andel av all tid som är pre-2013:", round(andel_tid_pre2013, 3))
print("Andel av topparna som är pre-2013:", round(andel_topp_pre2013, 3))

Andel av all tid som är pre-2013: 0.559
Andel av topparna som är pre-2013: 0.783


Nu hämtar vi data från fler stationer.

In [13]:
import pandas as pd
from pathlib import Path
from urllib.request import urlopen

BAS = "https://opendata-download-metobs.smhi.se/api/version/1.0"
KANDIDATER = {
    "Göteborg A": 71420, "Hanö A": 64020, "Hallands Väderö A": 52240,
    "Malexander A": 85050, "Harstena A": 87140, "Kettstaka A": 85460,
}

def hamta_och_sammanfatta(namn, station):
    fil = Path("../data/raw") / f"smhi_p21_{station}_corrected-archive.csv"
    if not fil.exists():
        url = f"{BAS}/parameter/21/station/{station}/period/corrected-archive/data.csv"
        with urlopen(url) as r:
            fil.write_bytes(r.read())

    with open(fil, encoding="utf-8-sig") as f:
        for nr, rad in enumerate(f):
            if rad.startswith("Datum;"):
                break

    d = pd.read_csv(fil, sep=";", skiprows=nr, usecols=[0, 1, 2, 3],
                     encoding="utf-8-sig")
    d.columns = ["datum", "tid", "byvind", "kvalitet"]
    d["tidpunkt"] = pd.to_datetime(d["datum"] + " " + d["tid"], utc=True)
    d = d.set_index("tidpunkt")[["byvind", "kvalitet"]]

    nominella = (d.index.max() - d.index.min()) / pd.Timedelta(hours=1) + 1
    return {
        "station": namn, "key": station, "rader": len(d),
        "start": d.index.min().date(), "slut": d.index.max().date(),
        "täckning": round(len(d) / nominella, 3),
        "andel_Y": round((d["kvalitet"] == "Y").mean(), 3),
        "p99_9": round(d["byvind"].quantile(0.999), 1),
    }

resultat = pd.DataFrame(hamta_och_sammanfatta(n, s) for n, s in KANDIDATER.items())
resultat

,station,key,rader,start,slut,täckning,andel_Y,p99_9
0,Göteborg A,71420,254288,1995-08-04,2026-08-01,0.936,0.559,21.0
1,Hanö A,64020,260895,1995-12-01,2026-08-01,0.970,0.556,30.2
2,Hallands Väderö A,52240,147547,2009-07-02,2026-08-01,0.985,0.207,26.2
3,Malexander A,85050,267239,1995-10-01,2026-08-01,0.989,0.559,19.6
4,Harstena A,87140,263599,1996-02-01,2026-08-01,0.986,0.557,23.0
5,Kettstaka A,85460,266555,1995-08-01,2026-08-01,0.981,0.559,18.6


In [14]:
import pandas as pd
from pathlib import Path

VALDA = {
    "Hanö A": 64020,
    "Harstena A": 87140,
    "Malexander A": 85050,
    "Kettstaka A": 85460,
}

def las_in_station(station):
    fil = Path("../data/raw") / f"smhi_p21_{station}_corrected-archive.csv"
    with open(fil, encoding="utf-8-sig") as f:
        for nr, rad in enumerate(f):
            if rad.startswith("Datum;"):
                break
    d = pd.read_csv(fil, sep=";", skiprows=nr, usecols=[0, 1, 2, 3],
                     encoding="utf-8-sig")
    d.columns = ["datum", "tid", "byvind", "kvalitet"]
    d["tidpunkt"] = pd.to_datetime(d["datum"] + " " + d["tid"], utc=True)
    return d.set_index("tidpunkt")[["byvind", "kvalitet"]]

def diagnostik(namn, station):
    d = las_in_station(station)
    nominella = (d.index.max() - d.index.min()) / pd.Timedelta(hours=1) + 1
    tackning = len(d) / nominella

    # Håller den administrativa brytpunkten 2013 även här?
    y = d["kvalitet"] == "Y"
    y_per_ar = y.groupby(d.index.year).mean()
    brytpunkt_haller = (y_per_ar[y_per_ar.index < 2013].min() > 0.9 and
                         y_per_ar[y_per_ar.index >= 2013].max() < 0.1)

    # Är extremvärden överrepresenterade i Y-perioden utöver ren tidsandel?
    pre2013 = d.index.year < 2013
    topp = d["byvind"] >= d["byvind"].quantile(0.999)
    andel_tid = pre2013.mean()
    andel_topp = topp[pre2013].sum() / topp.sum() if topp.sum() else float("nan")

    # Var ligger luckorna i kalenderåret? (MNAR-koll: klumpar de sig i stormsäsong?)
    full_axel = pd.date_range(d.index.min(), d.index.max(), freq="h", tz="UTC")
    saknas = full_axel.difference(d.index)
    luckor_per_manad = pd.Series(saknas.month).value_counts(normalize=True)
    andel_vinterluckor = luckor_per_manad.reindex([10, 11, 12, 1, 2, 3], fill_value=0).sum()

    return {
        "station": namn, "rader": len(d), "täckning": round(tackning, 3),
        "brytpunkt_2013_håller": brytpunkt_haller,
        "andel_tid_pre2013": round(andel_tid, 3),
        "andel_topp_pre2013": round(andel_topp, 3),
        "andel_luckor_okt_mar": round(andel_vinterluckor, 3),
    }

resultat = pd.DataFrame(diagnostik(n, s) for n, s in VALDA.items())
resultat

,station,rader,täckning,brytpunkt_2013_håller,andel_tid_pre2013,andel_topp_pre2013,andel_luckor_okt_mar
0,Hanö A,260895,0.970,True,0.556,0.616,0.582
1,Harstena A,263599,0.986,True,0.557,0.733,0.385
2,Malexander A,267239,0.989,True,0.559,0.624,0.682
3,Kettstaka A,266555,0.981,True,0.559,0.682,0.581


In [16]:
for namn, station in VALDA.items():
    d = las_in_station(station)
    full_axel = pd.date_range(d.index.min(), d.index.max(), freq="h", tz="UTC")
    saknas = full_axel.difference(d.index)
    topp_manad = pd.Series(saknas.month).value_counts().idxmax()
    print(namn, "-> toppmånad:", topp_manad)

Hanö A -> toppmånad: 11
Harstena A -> toppmånad: 6
Malexander A -> toppmånad: 10
Kettstaka A -> toppmånad: 12
